# Module 2: Comparative Analysis of Keras and PyTorch Models

This notebook compares the performance of a simple binary image classifier built using Keras and PyTorch.

We will:
- explain how thresholding works,
- print Keras performance metrics,
- explain the F1 score,
- print PyTorch performance metrics, and
- identify the false negatives in the PyTorch confusion matrix.

In [1]:
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
torch.manual_seed(SEED)

DATASET_PATH = './images_dataSAT'
if not os.path.exists(DATASET_PATH):
    alt_path = os.path.join('.', 'AI Capstone DL Projects', 'CNN Model Development', 'images_dataSAT')
    if os.path.exists(alt_path):
        DATASET_PATH = alt_path

print('Dataset path:', DATASET_PATH)
print('Dataset exists:', os.path.exists(DATASET_PATH))


def print_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    print('Accuracy: ', round(float(accuracy), 4))
    print('Precision: ', round(float(precision), 4))
    print('Recall: ', round(float(recall), 4))
    print('F1 Score: ', round(float(f1), 4))
    print('Confusion Matrix:\n', cm)

    return {
        'Accuracy': float(accuracy),
        'Precision': float(precision),
        'Recall': float(recall),
        'F1 Score': float(f1)
    }, cm

c:\Users\Furqan Khan\AppData\Local\miniconda3\envs\agent_env2\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\Furqan Khan\AppData\Local\miniconda3\envs\agent_env2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset path: .\AI Capstone DL Projects\CNN Model Development\images_dataSAT
Dataset exists: True


## Task 1: What does the code `preds = (preds > 0.5).astype(int).flatten()` do?

This code turns the model probability output into a binary class prediction.

- If the predicted probability is greater than 0.5, the result becomes 1.
- If it is less than or equal to 0.5, the result becomes 0.
- `astype(int)` converts the True/False result to integers.
- `.flatten()` makes the output a 1D array so it matches the ground-truth labels.

In short, it converts probabilities into clear class labels for evaluation.

In [2]:
# Example of thresholding probabilities
sample_probs = np.array([0.12, 0.67, 0.81, 0.44])
sample_preds = (sample_probs > 0.5).astype(int).flatten()

print('Sample probabilities:', sample_probs)
print('Thresholded predictions:', sample_preds)

Sample probabilities: [0.12 0.67 0.81 0.44]
Thresholded predictions: [0 1 1 0]


## Task 2: Print the performance metrics for the Keras model using `print_metrics`

We build a small CNN, train it for one epoch, make predictions, and then print the evaluation metrics.

In [3]:
keras_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    validation_split=0.2
)

train_generator = keras_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(64, 64),
    batch_size=16,
    class_mode='binary',
    subset='training',
    shuffle=True,
    seed=SEED
)

val_generator = keras_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(64, 64),
    batch_size=16,
    class_mode='binary',
    subset='validation',
    shuffle=False,
    seed=SEED
)

keras_model = Sequential([
    tf.keras.layers.Input(shape=(64, 64, 3)),
    Conv2D(16, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

keras_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = keras_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=1,
    verbose=0
)

keras_probs = keras_model.predict(val_generator, verbose=0)
keras_preds = (keras_probs > 0.5).astype(int).flatten()
keras_true = val_generator.classes

print('Keras model metrics:')
keras_metrics, keras_cm = print_metrics(keras_true, keras_preds)
print('Keras model training loss history:', history.history.get('loss'))

Found 4800 images belonging to 2 classes.
Found 1200 images belonging to 2 classes.
Keras model metrics:
Accuracy:  0.9675
Precision:  0.9947
Recall:  0.94
F1 Score:  0.9666
Confusion Matrix:
 [[597   3]
 [ 36 564]]
Keras model training loss history: [0.13049198687076569]


## Task 3: What is the significance of the F1 score?

The F1 score is important because it balances precision and recall.

- Precision tells us how many predicted positives are actually correct.
- Recall tells us how many true positives were found.
- F1 score combines both into a single value.

This makes it very useful when the dataset is imbalanced or when false negatives and false positives both matter.

## Task 4: Print the performance metrics for the PyTorch model using `print_metrics`

We create a simple CNN in PyTorch, train it for one epoch, and evaluate it on the validation set.

In [4]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 16 * 16, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Create PyTorch dataset and dataloaders
full_dataset = datasets.ImageFolder(
    root=DATASET_PATH,
    transform=transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor()
    ])
)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

pytorch_model = SimpleCNN()
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(pytorch_model.parameters(), lr=1e-3)

pytorch_model.train()
for images, labels in train_loader:
    optimizer.zero_grad()
    logits = pytorch_model(images).squeeze(1)
    loss = criterion(logits, labels.float())
    loss.backward()
    optimizer.step()

pytorch_model.eval()
y_true_pt = []
y_pred_pt = []

with torch.no_grad():
    for images, labels in val_loader:
        logits = pytorch_model(images).squeeze(1)
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).int().cpu().numpy()
        y_pred_pt.extend(preds)
        y_true_pt.extend(labels.numpy())

y_true_pt = np.asarray(y_true_pt)
y_pred_pt = np.asarray(y_pred_pt)

print('PyTorch model metrics:')
pt_metrics, pt_cm = print_metrics(y_true_pt, y_pred_pt)

PyTorch model metrics:
Accuracy:  0.9908
Precision:  0.9913
Recall:  0.9896
F1 Score:  0.9904
Confusion Matrix:
 [[619   5]
 [  6 570]]


## Task 5: What are the total number of false negatives in the confusion matrix in the PyTorch model evaluated above?

A false negative happens when the true label is 1, but the model predicts 0.

In a binary confusion matrix:
- row 1, column 0 = false negatives
- row 0, column 1 = false positives

We calculate it directly from the confusion matrix.

In [5]:
cm_pt = confusion_matrix(y_true_pt, y_pred_pt)
false_negatives = cm_pt[1, 0]

print('PyTorch confusion matrix:\n', cm_pt)
print('Total false negatives:', false_negatives)

PyTorch confusion matrix:
 [[619   5]
 [  6 570]]
Total false negatives: 6


## Final comparison

The two frameworks use similar ideas, but their code structure is different:

- Keras is compact and often easier for quick experimentation.
- PyTorch gives more control and is very popular in research and custom model work.

Both can produce binary classification results, and the same metric logic applies to both models.